# 08 — Grad-CAM (Figure 8)

**Tujuan.** Menguji Grad-CAM post-hoc pada modul Coordinate Attention, lalu kelak menampilkan RGB asli, heatmap, dan overlay Grade 1–4. Figure 8 hlm. 91512–91513 tidak menyebut layer target; `blocks.5.attention` adalah `ASSUMED`. Grad-CAM tidak masuk training loss. Output sintetis di bawah **bukan** citra fundus atau bukti lesi.

In [1]:
from pathlib import Path
import os, sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "reproduction/cadvit/model.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
ARTIFACT_ROOT = Path(os.environ["CAD_VIT_ARTIFACT_ROOT"]).expanduser().resolve() if os.environ.get("CAD_VIT_ARTIFACT_ROOT") else None
print("Artifact root:", ARTIFACT_ROOT if ARTIFACT_ROOT else "PENDING (set CAD_VIT_ARTIFACT_ROOT)")

Artifact root: PENDING (set CAD_VIT_ARTIFACT_ROOT)


In [2]:
import torch
from reproduction.cadvit.model import CADViTCandidate
from reproduction.cadvit.interpretability import gradcam, DEFAULT_TARGET_LAYER
torch.set_num_threads(1)
synthetic_model = CADViTCandidate().eval()
synthetic_result = gradcam(synthetic_model, torch.randn(1, 3, 64, 64))
print("SYNTHETIC_TEST target", DEFAULT_TARGET_LAYER,
      "heatmap", tuple(synthetic_result["heatmap"].shape),
      "range", (float(synthetic_result["heatmap"].min()), float(synthetic_result["heatmap"].max())))
print("PENDING real train/validation RGB + trusted checkpoint; no test image is opened")

SYNTHETIC_TEST target blocks.5.attention heatmap (64, 64) range (0.0, 0.0)
PENDING real train/validation RGB + trusted checkpoint; no test image is opened


In [3]:
from reproduction.cadvit.interpretability import load_controlled_best, plot_gradcam
from reproduction.cadvit.visualization import guarded_non_test_image, save_campaign_figure
if os.environ.get("CAD_VIT_RUN_GRADCAM") == "1":
    from PIL import Image, ImageOps
    import numpy as np
    from reproduction.cadvit.evaluation import file_sha256
    from reproduction.cadvit.preprocessing import preprocess_fundus
    from IPython.display import display
    if ARTIFACT_ROOT is None:
        raise ValueError("CAD_VIT_ARTIFACT_ROOT is required for checkpoint provenance")
    image_path = os.environ["CAD_VIT_AUDIT_IMAGE"]
    dataset_root = os.environ["CAD_VIT_DATA_ROOT"]
    manifest_path = os.environ["CAD_VIT_AUDIT_MANIFEST"]
    source_split = os.environ.get("CAD_VIT_AUDIT_SPLIT", "train")
    campaign = os.environ.get("CAD_VIT_CAMPAIGN", "balanced_v1")
    variant = os.environ.get("CAD_VIT_VARIANT", "aptos_r16")
    checked_path, row = guarded_non_test_image(image_path, dataset_root, manifest_path, source_split)
    model, provenance = load_controlled_best(ARTIFACT_ROOT, campaign, variant, trust_checkpoint=True)
    if file_sha256(manifest_path) != provenance["manifest_sha256"]:
        raise ValueError("Audit manifest and checkpoint fingerprint differ")
    with Image.open(checked_path) as image:
        original = np.asarray(ImageOps.exif_transpose(image).convert("RGB"))
        pixels = preprocess_fundus(image)
    tensor = torch.from_numpy(np.array(pixels, copy=True)).permute(2, 0, 1).float()[None] / 255
    result = gradcam(model, tensor)
    figure = plot_gradcam(original, result, int(row["label"]), provenance)
    display(figure)
    if os.environ.get("CAD_VIT_SAVE_FIGURES") == "1":
        stem = "gradcam_" + "".join(ch if ch.isalnum() else "_" for ch in row["image_id"])
        output = save_campaign_figure(figure, ARTIFACT_ROOT, campaign, variant, stem)
        print("Saved", output)
else:
    print("PENDING real Grad-CAM: set CAD_VIT_RUN_GRADCAM=1 only with verified train/val image and trusted checkpoint")

PENDING real Grad-CAM: set CAD_VIT_RUN_GRADCAM=1 only with verified train/val image and trusted checkpoint


**Protokol real-image kemudian.** Verifikasi path terhadap manifest train/validation memakai `guarded_non_test_image`; load checkpoint dengan `load_controlled_best(..., trust_checkpoint=True)`; masukkan **preprocess_fundus V1** ke model, gunakan RGB asli hanya untuk display. `plot_gradcam` menampilkan tiga panel dan metadata checkpoint. Simpan hasil ke folder `figures/` kampanye dengan `save_campaign_figure`. Notebook tidak menjalankan tahap itu secara otomatis. Panas heatmap bukan validasi klinis lesi.